# A simple queue model

Each time step has a chance of one arrival, then a service capacity of zero or one. This is a deliberately simplified school-canteen model. It omits groups, varying service times and people abandoning the queue. Predict which assumptions could underestimate waiting.

In [1]:
from random import Random

def simulate(arrival_probability, service_capacity, steps, seed):
    if not 0 <= arrival_probability <= 1 or service_capacity not in (0, 1) or steps < 0:
        raise ValueError("Invalid model parameter")
    rng = Random(seed)
    queue = 0
    history = []
    for _ in range(steps):
        queue += int(rng.random() < arrival_probability)
        queue = max(0, queue - service_capacity)
        history.append(queue)
    return history

assert simulate(0, 1, 5, 1) == [0] * 5
assert simulate(1, 0, 5, 1) == [1, 2, 3, 4, 5]
print(simulate(0.6, 0, 10, 7))


[1, 2, 2, 3, 4, 5, 6, 7, 8, 9]


## First critique

With at most one arrival and guaranteed service for one person every step, the end-of-step queue is always zero. The model cannot show congestion under that configuration. A plausible graph is not proof of a useful model. What should change to represent variable service?

In [2]:
def variable_service(arrival_probability, service_probability, steps, seed):
    if not 0 <= arrival_probability <= 1 or not 0 <= service_probability <= 1 or steps < 0:
        raise ValueError("Invalid model parameter")
    rng = Random(seed)
    queue = 0
    history = []
    for _ in range(steps):
        arrives = rng.random() < arrival_probability
        serves = rng.random() < service_probability
        queue += int(arrives)
        queue = max(0, queue - int(serves))
        history.append(queue)
    return history

assert variable_service(0, 0, 5, 3) == [0] * 5
assert variable_service(1, 0, 5, 3) == [1, 2, 3, 4, 5]
assert variable_service(1, 1, 5, 3) == [0] * 5
print(variable_service(0.7, 0.5, 10, 7))


[0, 0, 0, 1, 1, 1, 2, 2, 3, 3]


## Compare scenarios

Use ten fixed seeds for each scenario. The same seed recreates a sequence; it does not make the model accurate. Predict which service probability gives a larger mean final queue. Change only that parameter.

In [3]:
for service_probability in [0.5, 0.8]:
    finals = [variable_service(0.7, service_probability, 100, seed)[-1] for seed in range(10)]
    print(service_probability, finals, sum(finals) / len(finals))


0.5 [15, 20, 18, 10, 5, 32, 18, 20, 22, 20] 18.0
0.8 [0, 1, 1, 1, 0, 3, 0, 0, 1, 0] 0.7


## Report

State the inputs, update order, comparison result, variation across runs and limitations. The final queue length is not the average waiting time; do not label it as one. An extension can track the arrival time of each individual, making waiting time a separate measured output.